### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors care about the Mean Absolute Error (MAE). Seniors care about the **Predictive Variance**. In production, we don't just want the prediction; we want the model to tell us how 'surprised' it is by the current input. If the input is far from the training data, a Bayesian model's variance will explode, alerting us to a potential OOD (Out-Of-Distribution) failure.

**Mental Model:** 
- **Linear Regression:** Finding the single 'best' line.
- **Bayesian Linear Regression:** Every possible line exists simultaneously, weighted by how well it fits the data (The Posterior).
- **Gaussian Processes:** Instead of lines, every possible smooth function exists simultaneously.

**Common Junior Pitfall:** Confusing 'Model Uncertainty' (Epistemic) with 'Data Noise' (Aleatoric). Bayesian methods primarily help us quantify what the model *doesn't know* because it lacks data (Epistemic), which is what we need for safety-critical systems.

---


## 1. Bayesian Linear Regression — Full Posterior Inference

Frequentist regression gives a point estimate for weights $w$. Bayesian regression gives a distribution $P(w|D)$.

### The Closed-Form Math
With a Gaussian prior $P(w) = N(0, \alpha^{-1}I)$ and Gaussian likelihood $P(y|X, w) = N(Xw, \beta^{-1}I)$, the posterior is:
$$\Sigma_N^{-1} = \alpha I + \beta X^T X$$
$$m_N = \beta \Sigma_N X^T y$$


## 📑 Table of Contents

  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1. Bayesian Linear Regression — Full Posterior Inference](#1-bayesian-linear-regression-full-posterior-inference)
  * [The Closed-Form Math](#the-closed-form-math)
* [2. Gaussian Processes — Infinite-Dimensional Bayesian Regression](#2-gaussian-processes-infinite-dimensional-bayesian-regression)
* [3. Bayesian Hyperparameter Optimization](#3-bayesian-hyperparameter-optimization)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: Point Estimate vs Posterior](#q1-point-estimate-vs-posterior)
  * [Q2: GP predictive variance](#q2-gp-predictive-variance)
* [🔨 Practical Practice](#practical-practice)
  * [Tier 1: Beginner](#tier-1-beginner)
  * [Tier 2: Intermediate](#tier-2-intermediate)
  * [Tier 3: Advanced](#tier-3-advanced)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class BayesianLinearRegression:
    def __init__(self, alpha=1.0, beta=1.0):
        self.alpha, self.beta = alpha, beta
        
    def fit(self, X, y):
        # closed form posterior parameters
        self.S_inv = self.alpha * np.eye(X.shape[1]) + self.beta * X.T @ X
        self.S = np.linalg.inv(self.S_inv)
        self.m = self.beta * self.S @ X.T @ y
        
    def predict(self, X):
        y_mean = X @ self.m
        # predictive variance includes noise term
        y_var = 1/self.beta + np.sum(X @ self.S * X, axis=1)
        return y_mean, np.sqrt(y_var)

# Toy Data: Noisy linear points concentrated in the middle
X = np.linspace(-1, 1, 10).reshape(-1, 1)
X_bias = np.c_[np.ones(10), X]
y = 2 * X.squeeze() + np.random.normal(0, 0.2, 10)

model = BayesianLinearRegression(alpha=2.0, beta=25.0)
model.fit(X_bias, y)

X_test = np.linspace(-3, 3, 100).reshape(-1, 1)
X_test_bias = np.c_[np.ones(100), X_test]
mu, std = model.predict(X_test_bias)

plt.scatter(X, y, color='red', label='Training Data')
plt.plot(X_test, mu, color='blue', label='Posterior Mean')
plt.fill_between(X_test.squeeze(), mu-2*std, mu+2*std, alpha=0.2, color='blue', label='2σ Uncertainty')
plt.title("Bayesian Linear Regression: Uncertainty Widens Away from Data")
plt.legend()
plt.show()

"""
What just happened?
We derived the posterior distribution of the weights. Notice how the blue 
shaded region (uncertainty) is tight where we have data, but explodes as 
we move toward X=-3 or X=3. The model 'doesn't know' what happens there.
"""

## 2. Gaussian Processes — Infinite-Dimensional Bayesian Regression

Instead of a prior over weights $P(w)$, a GP defines a prior over **Entire Functions** $P(f)$. 
Every finite collection of points $(x_1, \dots, x_n)$ is distributed as a Joint Normal distribution:
$$P(f(x_1), \dots, f(x_n)) = N(0, K)$$
Where $K_{ij}$ is the **RBF (Radial Basis Function)** kernel: $k(x_i, x_j) = \exp(-\frac{||x_i-x_j||^2}{2l^2})$.

In [ ]:
def rbf_kernel(a, b, length_scale=1.0):
    sq_dist = np.sum(a**2, 1).reshape(-1, 1) + np.sum(b**2, 1) - 2 * np.dot(a, b.T)
    return np.exp(-0.5 * sq_dist / length_scale**2)

def gp_predict(X_train, y_train, X_test, kernel_fn, noise=0.1):
    K = kernel_fn(X_train, X_train) + noise**2 * np.eye(len(X_train))
    K_s = kernel_fn(X_train, X_test)
    K_ss = kernel_fn(X_test, X_test) + 1e-8 * np.eye(len(X_test))
    K_inv = np.linalg.inv(K)
    
    mu_s = K_s.T @ K_inv @ y_train
    cov_s = K_ss - K_s.T @ K_inv @ K_s
    return mu_s, np.sqrt(np.diag(cov_s))

X_gp = np.array([0.5, 1.5, 2.5, 4.5, 5.5]).reshape(-1, 1)
y_gp = np.sin(X_gp).flatten()
X_s = np.linspace(0, 10, 100).reshape(-1, 1)

mu_gp, std_gp = gp_predict(X_gp, y_gp, X_s, rbf_kernel)

plt.plot(X_s, mu_gp, color='green', label='GP Mean')
plt.fill_between(X_s.flatten(), mu_gp-2*std_gp, mu_gp+2*std_gp, alpha=0.1, color='green')
plt.scatter(X_gp, y_gp, color='black', label='Observations')
plt.title("Gaussian Process Regression: Priors over Functions")
plt.legend()
plt.show()

"""
What just happened?
The GP maps correlations. Points close in X-space are forced to be close in 
Y-space based on the RBF kernel. Where data is missing (e.g., X=8), the GP 
reverts to its prior (mean=0) and the variance expands back to 1.0.
"""

## 3. Bayesian Hyperparameter Optimization

Standard Grid/Random search is "blind." It doesn't use previous results to pick the next candidates.
**Bayesian Optimization (BO)** fits a GP to the (Hyperparams → Validation Accuracy) surface and uses an **Acquisition Function** (like Expected Improvement) to find the global optimum with very few trials.

📈 **Production Signal:** Using BO to tune a 1-billion parameter LLM can save thousands of dollars in compute time compared to brute-force grid search.

In [ ]:
try:
    from skopt import BayesSearchCV
    import xgboost as xgb
except ImportError:
    print("Installing requirements...")
    !pip install -q scikit-optimize xgboost
    from skopt import BayesSearchCV
    import xgboost as xgb

from sklearn.datasets import load_breast_cancer
X_cancer, y_cancer = load_breast_cancer(return_X_y=True)

# Define Search Space
opt = BayesSearchCV(
    xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    {
        'learning_rate': (0.01, 1.0, 'log-uniform'),
        'max_depth': (1, 10),
        'n_estimators': (10, 100)
    },
    n_iter=10,
    cv=3
)

opt.fit(X_cancer, y_cancer)
print(f"Best Parameters Found: {opt.best_params_}")
print(f"Best Score: {opt.best_score_:.4f}")

"""
What just happened?
Instead of randomly guessing, the GP modeled the XGBoost accuracy surface. 
It automatically moved toward learning rates and depths that 'looked promising,' 
finding a high-performing configuration in just 10 trials.
"""

---
## ✅ Knowledge Check

### Q1: Point Estimate vs Posterior
<details><summary>👉 Answer</summary>
A point estimate gives the 'most likely' single line. The Posterior gives weights as distributions, allowing the model to calculate its own 'confidence interval' for every prediction. 
</details>

### Q2: GP predictive variance
<details><summary>👉 Answer</summary>
Prediction variance in a GP is lowest at the observed data points and highest in 'data deserts' far from training data. This makes GPs ideal for Active Learning and Optimization where we want to sample where we are most uncertain.
</details>


---
## 🔨 Practical Practice

### Tier 1: Beginner
1. Manually calculate the Likelihood $P(D|w)$ for 3 coin flips (Heads=1, Tails=0) using a Bernoulli distribution.
2. Modify the Bayesian Linear Regression code to increase the 'noise' parameter and observe how the uncertainty bands change.

### Tier 2: Intermediate
1. **RBF Length Scale:** Modify the GP length scale $l$ from 0.1 to 5.0. Explain why a large length scale produces 'stiffer' functions and why a tiny length scale produces 'jittery' ones.
2. **The Acquisition Function:** Using `skopt.acquisition`, calculate the Expected Improvement for a 1D GP model at 10 test points. Identify where the algorithm would sample next. 

### Tier 3: Advanced
1. **Gaussian Conjugacy:** Prove (on paper or in markdown) that the product of two Gaussian distributions (Prior and Likelihood) results in another Gaussian (The Conjugacy property). 
2. **Multi-Fidelity BO:** Design an optimization loop where you use the BO findings from a small subset of the data to 'hot-start' the search on the full dataset. 
